# NSTT-Lite Plan 4 — full-scale Colab run

Runs plan4.md Steps 1-5 in order. Each code cell prints what to screenshot for Step 6 -- take the screenshot before moving to the next cell, don't batch them at the end.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Restart session`.

## 0. Mount Drive, clone/pull, install, GPU check

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
PROJECT_ROOT = '/content/drive/MyDrive/NSTT-Lite'
if os.path.exists(PROJECT_ROOT):
    !cd {PROJECT_ROOT} && git pull
else:
    !git clone https://github.com/Rbimochan/NSTT-Lite.git {PROJECT_ROOT}
    !cd {PROJECT_ROOT} && git checkout feature/plan2-plan3-execution
%cd {PROJECT_ROOT}

In [ ]:
!pip install -q -r requirements.txt
# Restart runtime after this cell if you see a red ResolutionImpossible error, then re-run from here.

In [ ]:
import torch, transformers, datasets
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('torch', torch.__version__, 'transformers', transformers.__version__, 'datasets', datasets.__version__)
# SCREENSHOT NOW: this cell's output (Plan 4 Step 6 -- GPU runtime type + library versions).

## Data check
Corpus (`data/openslr54_ne/`) and manifests (`data/manifests/`) must already be on Drive from the local work -- this notebook does not re-download the corpus.

In [ ]:
!wc -l data/manifests/*.jsonl
!ls data/openslr54_ne/ 2>&1 | head -5

## Step 1: full-scale training (AdamW, FP16, batch 2, grad-accum 4, max 5 epochs / 2000 steps, early stopping patience 2)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir models/finetuned_full/runs
# Leave this cell's TensorBoard panel open in a browser tab while the next cell trains --
# it updates live. SCREENSHOT the loss/WER curves once training finishes (Step 6).

In [ ]:
!python scripts/run_colab_training.py

## Step 2: re-evaluate on the FULL test split, baseline vs. fine-tuned, plus matched-step ablation

In [ ]:
!NSTT_CHECKPOINT_DIR=models/finetuned_full NSTT_MANIFEST_DIR=data/manifests NSTT_EVAL_NAME=finetuned_full \
  python scripts/run_full_test_eval.py

In [ ]:
# Zero-shot baseline on the full test split too, for a fair comparison at matched scale.
!NSTT_CHECKPOINT_DIR=openai/whisper-small NSTT_MANIFEST_DIR=data/manifests NSTT_EVAL_NAME=baseline_full \
  python scripts/run_full_test_eval.py

In [ ]:
!python scripts/make_leaky_manifests.py
!python scripts/run_colab_ablation_training.py

In [ ]:
!NSTT_CHECKPOINT_DIR=models/finetuned_leaky_full NSTT_MANIFEST_DIR=data/manifests_leaky NSTT_EVAL_NAME=leaky_full \
  python scripts/run_full_test_eval.py
# Now compute the matched-step WER gap: reports/eval_finetuned_full.json vs reports/eval_leaky_full.json

## Step 3: gender classification on the full-scale encoder

In [ ]:
!NSTT_CHECKPOINT_DIR=models/finetuned_full python scripts/run_gender_classifier.py

## Step 4: refresh error analysis and own-voice test

In [ ]:
!NSTT_CHECKPOINT_DIR=models/finetuned_full python scripts/run_error_analysis.py
# Own-voice test needs data/iphone_recordings/ and afconvert (macOS-only) -- 
# if running purely on Colab, skip this cell and re-run run_own_voice_test.py locally
# pointed at the Drive-synced models/finetuned_full checkpoint instead.

## Step 5: redeploy against the full-scale checkpoint

In [ ]:
!pip install -q ctranslate2 faster-whisper
!NSTT_CHECKPOINT_DIR=models/finetuned_full NSTT_CT2_DIR=models/finetuned_full_ct2 \
  python scripts/convert_ctranslate2.py

In [ ]:
!pip install -q streamlit
!NSTT_CHECKPOINT_DIR=models/finetuned_full streamlit run scripts/app.py &
# Use Colab's port-forwarding (e.g. via ngrok or `google.colab.output.serve_kernel_port_as_window`)
# to open the app in a browser tab, upload a real sample, and SCREENSHOT the running app (Step 6).